# Netlist State-Space Validation

This example validates the `NetlistStateSpace` block against an equivalent PathSim implementation of an LC filter with a resistive load.

Two circuit descriptions are used. The first netlist contains the load explicitly as a resistor. The second represents the same load using a behavioral current source. Both netlists are passed directly to `NetlistStateSpace` without requiring external `.net` files.

The simulation compares the PathSim implementation and the imported netlist models using the recorded capacitor voltage and load current signals.


In [ ]:
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import (
    Adder,
    Amplifier,
    Integrator,
    PulseSource,
    Scope
)
from pathsim.solvers import EUB
from pathsim_rf.netlist_to_statespace import NetlistStateSpace


In [ ]:
Rload = 50
Cfilter = 50e-6
Lfilter = 0.5e-3


## Netlists

The first netlist contains the 50 Ω load explicitly as `Rload`.


In [ ]:
filter_netlist = """
*validation netlist for pathsim
.param FREQ = 9k
.param DUTY = 0.5
.param V_LOW = 0
.param V_HIGH = 1
.param T_PER = {1/FREQ}
.param T_ON = {T_PER * DUTY}
.param T_RISE = 10n
.param T_FALL = 10n

V1 in 0 PULSE({V_LOW} {V_HIGH} 0 {T_RISE} {T_FALL} {T_ON} {T_PER})
Lfilter in n1 0.5e-3
Cfilter n1 0 50e-6
Rload n1 0 50
.tran 0.00001 0.2 0 0.000001
.end
"""


The second netlist represents the load using a behavioral current source.


In [ ]:
filter_without_load_netlist = """
*validation netlist for pathsim
.param FREQ = 9k
.param DUTY = 0.5
.param V_LOW = 0
.param V_HIGH = 1
.param T_PER = {1/FREQ}
.param T_ON = {T_PER * DUTY}
.param T_RISE = 10n
.param T_FALL = 10n

V1 in 0 PULSE({V_LOW} {V_HIGH} 0 {T_RISE} {T_FALL} {T_ON} {T_PER})
Bload n1 0 I=V(n1)/50
Lfilter in n1 0.5e-3
Cfilter n1 0 50e-6
.tran 0.00001 0.2 0 0.00001
.end
"""


## Blocks


In [ ]:
# Sources
pulsesource = PulseSource(
    T=1/9000,
    amplitude=1,
    duty=0.5,
    t_rise=10e-9, 
    t_fall=10e-9
)

state_space = NetlistStateSpace(
    filter_netlist,
    output_voltages=["n1"],
    output_currents=["Rload"],
)

state_space_bis = NetlistStateSpace(
    filter_without_load_netlist,
    output_voltages=["n1"],
    output_currents=["Lfilter"],
)

# Dynamic
integrator = Integrator(initial_value=0)
block_2 = Integrator()

# Algebraic
adder = Adder(operations="+-")
amplifier = Amplifier(gain=1/Lfilter)
block_5 = Adder(operations="+-")
block_6 = Amplifier(gain=1/Cfilter)
block_7 = Amplifier(gain=1/Rload)
block_8 = Amplifier(gain=1/Rload)

# Recording
scope = Scope(
    sampling_period=1/100000,
    labels=[
        'V(pwm)',
        'V(Cfull_pathsim)',
        'V(C_net_imported)',
        'V(Bload)'
    ]
)

blocks = [
    pulsesource,
    integrator,
    block_2,
    adder,
    amplifier,
    block_5,
    block_6,
    block_7,
    scope,
    state_space,
    state_space_bis,
    block_8
]


## Connections


In [ ]:
conn_0 = Connection(pulsesource[0], adder[0], state_space[0], state_space_bis[0])
conn_1 = Connection(adder[0], integrator[0])
conn_2 = Connection(integrator[0], amplifier[0])
conn_3 = Connection(amplifier[0], block_5[0])
conn_4 = Connection(block_5[0], block_2[0])
conn_5 = Connection(block_2[0], block_6[0])
conn_6 = Connection(block_6[0], block_7[0])
conn_7 = Connection(block_7[0], block_5[1])
conn_8 = Connection(block_6[0], adder[1])
conn_9 = Connection(pulsesource[0], scope[0])
conn_10 = Connection(block_6[0], scope[1])
conn_11 = Connection(state_space[0], scope[2])
conn_12 = Connection(state_space_bis[0], block_8[0], scope[3])
conn_13 = Connection(block_8[0], state_space_bis[1])

connections = [
    conn_0,
    conn_1,
    conn_2,
    conn_3,
    conn_4,
    conn_5,
    conn_6,
    conn_7,
    conn_8,
    conn_9,
    conn_10,
    conn_11,
    conn_12,
    conn_13
]


## Simulation


In [ ]:
sim = Simulation(
    blocks,
    connections,
    Solver=EUB,
    dt=0.5e-6,
    dt_min=1e-16,
    dt_max=1e-6,
    tolerance_lte_rel=1e-6,
    tolerance_lte_abs=1e-10,
    tolerance_fpi=1e-10,
)


## Results


In [ ]:
sim.run(duration=0.05)

sim.plot()
plt.show()
